In [1]:
import os
os.environ["AI2POT_PATH"] = "/data/home/liuhanyu/mycode/AI2Pot/"
os.environ["AI2POT_TEST_DATA_PATH"] = os.path.join(os.environ.get("AI2POT_PATH"),
                                                   "test",
                                                   "test_data",
                                                   "XYZ")
extxyz_path: str = os.path.join(os.environ.get("AI2POT_TEST_DATA_PATH"),
                                               "11_NEP_potential_PbTe",
                                               "train_m.xyz")

from typing import List
import torch
from ai2pot.data.mlffdatamodule import ExtxyzDataModule
from ai2pot.models.potential_train import LitLinearMtp
import lightning as L
from lightning.pytorch.loggers import TensorBoardLogger

# 1. Start training a `Lienar MTP model`

In [2]:
save_dir: str = os.path.join(os.environ.get("AI2POT_PATH"),
                                            "lightning_logs")
name: str = "linear_mtp"

max_epochs: int = 500
accelerator: str = "cpu"
devices: int = 1

trainset_path: str = extxyz_path
rcut: float = 6.0
umax_num_neigh_atoms: int = 200
pbc_xyz: List[bool] = [True, True, True]
sort: bool = True
torch_float_dtype: torch._C.dtype = torch.float32
has_virial: bool = False


mtp_level: int = 16
type_map: List[int] = [52, 82]
chebyshev_size: int = 8
rmax: float = rcut
rmin: float = 0.0
umax_num_neighs: int = umax_num_neigh_atoms
zbl_rmax: float = 0.0
zbl_rmin: float = 0.0

torch.set_num_threads(16)
torch.manual_seed(4231)

In [3]:
tb_logger: TensorBoardLogger = TensorBoardLogger(save_dir=save_dir,
                                                 name=name)
trainer: L.Trainer = L.Trainer(max_epochs=max_epochs,
                               accelerator=accelerator,
                               devices=devices,
                               logger=tb_logger,
                               limit_val_batches=0)
datamodule: ExtxyzDataModule = ExtxyzDataModule(trainset_path=trainset_path,
                                                validset_path=trainset_path,
                                                rcut=rcut,
                                                umax_num_neigh_atoms=umax_num_neigh_atoms,
                                                pbc_xyz=pbc_xyz,
                                                sort=sort,
                                                torch_float_dtype=torch_float_dtype,
                                                has_virial=has_virial)
lit_linear_mtp: LitLinearMtp = LitLinearMtp(mtp_level=mtp_level,
                                            type_map=type_map,
                                            chebyshev_size=chebyshev_size,
                                            rmax=rmax,
                                            rmin=rmin,
                                            umax_num_neighs=umax_num_neigh_atoms,
                                            fit_virial=has_virial,
                                            zbl_rmax=zbl_rmax,
                                            zbl_rmin=zbl_rmin,
                                            zbl_cks_list=None,
                                            zbl_dks_list=None,
                                            torch_float_dtype=torch_float_dtype,
                                            lr_start=1e-1,
                                            lr_end=1e-3)

trainer.fit(model=lit_linear_mtp,
            datamodule=datamodule)

/data/home/liuhanyu/miniconda3/envs/mlff_2/lib/python3.11/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /data/home/liuhanyu/miniconda3/envs/mlff_2/lib/pytho ...
You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


/data/home/liuhanyu/miniconda3/envs/mlff_2/lib/python3.11/site-packages/lightning/pytorch/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
/data/home/liuhanyu/miniconda3/envs/mlff_2/lib/python3.11/site-packages/lightning/pytorch/core/optimizer.py:378: Found unsupported keys in the optimizer configuration: {'gradient_clip_algorithm', 'gradient_clip_val'}

  | Name  | Type      | Params | Mode 
--------------------------------------------
0 | model | LinearMtp | 222    | train
--------------------------------------------
222       Trainable params
0         Non-trainable params
222       Total params
0.001     Total estimated model params size (MB)
1         Modules in train mode
0         Modules in eval mode
/data/home/liuhanyu/miniconda3/envs/mlff_2/lib/python3.11/site-packages/lightning/pytorch/core/saving.py:363: Skipping 'torch_float_dtype' parameter because it is not possible to safely dump to YAML.
/data/home/liuhanyu/minicon

/data/home/liuhanyu/mycode/AI2Pot/test/test_data/XYZ/11_NEP_potential_PbTe/train_m.xyz
Epoch 0:   4%|▍         | 1/25 [00:00<00:06,  3.86it/s, v_num=13, train_mse_step=3.03e+3]tensor([  74.1373,   40.6776,  -29.4962,  -73.0388,  -50.6514,   17.4446,
          69.7760,   59.1165,    8.7522,    4.8005,   -3.4858,   -8.6235,
          -5.9737,    2.0691,    8.2413,    6.9705, -273.6739, -150.1231,
         108.9627,  269.6394])
Epoch 0:   8%|▊         | 2/25 [00:00<00:04,  4.70it/s, v_num=13, train_mse_step=5.65e+3]tensor([-8270.2383, -4347.9424,  3625.1733,  8006.8521,  4770.9712, -2727.2864,
        -7266.0254, -4829.7832, -2463.7632, -1290.7891,  1088.9342,  2385.3909,
         1403.9545,  -833.8933, -2164.9197, -1410.9116,  2847.8521,  1494.1592,
        -1254.5542, -2757.6199])
Epoch 0:  12%|█▏        | 3/25 [00:00<00:04,  5.02it/s, v_num=13, train_mse_step=65.50]  tensor([-397.8091, -218.2184,  158.3904,  391.9550,  271.6124,  -93.9167,
        -374.5602, -316.9731,   85.5289,   46.


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

# 2. Tensorboard logger

# 3. Calculate `RMSE` of `energy`, `force`, `virial`